In [ ]:
import sys
from pathlib import Path
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # raíz del proyecto (el notebook está en notebooks/)

%load_ext autoreload
%autoreload 2
import pandas as pd
from src.monitorizacion.simulacion import predecir_dia_sim, TABLA_FEATURES
from src.features.feature_sets import get_feature_sets
from src.orquestacion.reentrenamiento import entrenar
from src.monitorizacion.evaluacion import metricas_dia
from src.monitorizacion.simulacion import politica_congelado, politica_mensual, politica_semanal, simular
from src.monitorizacion.simulacion import ROOT
import matplotlib.pyplot as plt


In [ ]:


df = pd.read_parquet(TABLA_FEATURES)
feats = get_feature_sets(df)["predictivo"]

modelo = entrenar.fn(df[df["fecha"] < "2025-01-01"])   # modelo entrenado con datos hasta 2024
dia_sim = predecir_dia_sim(modelo, df, pd.Timestamp("2025-01-15"), feats)
dia_sim
metricas_dia(dia_sim)

In [ ]:

print(politica_congelado(pd.Timestamp("2025-02-01"), None, []))   # False
print(politica_mensual(pd.Timestamp("2025-02-01"), None, []))     # True
print(politica_mensual(pd.Timestamp("2025-02-15"), None, []))     # False

In [ ]:
sim, reentrenos = simular(df, politica_mensual, "2025-01-01", "2025-03-31")
print(len(sim), reentrenos)

In [ ]:


print(politica_semanal(pd.Timestamp("2026-10-05"), None, []))   # lunes  → True
print(politica_semanal(pd.Timestamp("2026-10-07"), None, []))   # miércoles → False

In [ ]:
from src.monitorizacion.simulacion import politica_congelado

sim_cong, re_cong = simular(df, politica_congelado, "2025-01-01", "2026-10-05")
sim_mens, re_mens = simular(df, politica_mensual,   "2025-01-01", "2026-10-05")

print("congelado:", len(sim_cong), "filas,", len(re_cong), "reentrenos")
print("mensual:  ", len(sim_mens), "filas,", len(re_mens), "reentrenos")

In [ ]:

sim_cong.to_parquet(ROOT / "reports" / "sim_congelado.parquet", index=False)
sim_mens.to_parquet(ROOT / "reports" / "sim_mensual.parquet", index=False)

In [ ]:
from src.monitorizacion.evaluacion import calcular_metricas

m_cong = calcular_metricas(sim_cong)
m_mens = calcular_metricas(sim_mens)

print(m_cong[["mae_modelo", "mae_naive"]].mean().round(2))
print(m_mens[["mae_modelo", "mae_naive"]].mean().round(2))

In [ ]:


fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(m_cong.index, m_cong["mae_modelo_7d"], label="congelado")
ax.plot(m_mens.index, m_mens["mae_modelo_7d"], label="mensual")
ax.plot(m_mens.index, m_mens["mae_naive_7d"], label="naive", color="grey", alpha=0.6)
ax.set_ylabel("MAE media 7 días (€/MWh)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
ratio_mens = m_mens["mae_modelo_7d"] / m_mens["mae_naive_7d"]
ratio_cong = m_cong["mae_modelo_7d"] / m_cong["mae_naive_7d"]

print("mensual:  ", ratio_mens.describe(percentiles=[.5, .9, .95]).round(2).to_dict())
print("congelado:", ratio_cong.describe(percentiles=[.5, .9, .95]).round(2).to_dict())
print("% semanas perdiendo contra el naive → mensual:", (ratio_mens > 1).mean().round(2),
      "| congelado:", (ratio_cong > 1).mean().round(2))

In [ ]:
from src.monitorizacion.simulacion import politica_trigger

sim_trig, re_trig = simular(df, politica_trigger, "2026-05-01", "2026-09-30")
print(len(re_trig), [d.date() for d in re_trig])

In [ ]:
m_trig = calcular_metricas(sim_trig)

# señales diarias de la simulación trigger
err = sim_trig["prediccion"] - sim_trig["precio_real"]
sesgo_dia = err.groupby(sim_trig["fecha_objetivo"]).mean()          # >0 sobrepredice, <0 infrapredice
sesgo_7d = sesgo_dia.rolling("7D", min_periods=5).mean()
ratio_7d = m_trig["mae_modelo_7d"] / m_trig["mae_naive_7d"]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(m_trig["mae_modelo_7d"], label="modelo"); axes[0].plot(m_trig["mae_naive_7d"], label="naive", color="grey")
axes[0].set_ylabel("MAE 7d (€)"); axes[0].legend()
axes[1].plot(ratio_7d); axes[1].axhline(1.3, color="red", ls="--"); axes[1].set_ylabel("ratio modelo/naive")
axes[2].plot(sesgo_7d); axes[2].axhline(0, color="black", lw=0.8); axes[2].set_ylabel("sesgo 7d (€)")
for ax in axes:
    for d in re_trig:
        ax.axvline(d, color="green", alpha=0.5)
    ax.grid(alpha=0.3)
plt.show()

In [ ]:
err_m = sim_mens["prediccion"] - sim_mens["precio_real"]
sesgo_m = err_m.groupby(sim_mens["fecha_objetivo"]).mean().rolling("7D", min_periods=5).mean()

print(sesgo_m.abs().describe(percentiles=[.5, .9, .95]).round(1).to_dict())
print("% días con |sesgo 7d| > 8 €:",  (sesgo_m.abs() > 8).mean().round(2))
print("% días con |sesgo 7d| > 10 €:", (sesgo_m.abs() > 10).mean().round(2))

In [ ]:
from src.monitorizacion.simulacion import politica_semanal, politica_trigger

sim_sem, re_sem = simular(df, politica_semanal, "2025-01-01", "2026-10-05")
sim_sem.to_parquet(ROOT / "reports" / "sim_semanal.parquet", index=False)

sim_trig, re_trig = simular(df, politica_trigger, "2025-01-01", "2026-10-05")
sim_trig.to_parquet(ROOT / "reports" / "sim_trigger.parquet", index=False)

print("semanal:", len(re_sem), "reentrenos")
print("trigger:", len(re_trig), "reentrenos →", [d.date() for d in re_trig])

In [ ]:
politicas = {
    "congelado": sim_cong,
    "mensual":   sim_mens,
    "semanal":   sim_sem,
    "trigger":   sim_trig,
}
n_reentrenos = {"congelado": 1, "mensual": 22, "semanal": len(re_sem), "trigger": len(re_trig)}

filas = {}
for nombre, sim in politicas.items():
    m = calcular_metricas(sim)
    err = sim["prediccion"] - sim["precio_real"]
    verano = sim["fecha_objetivo"] >= "2026-07-01"
    filas[nombre] = {
        "reentrenos":        n_reentrenos[nombre],
        "MAE modelo":        m["mae_modelo"].mean(),
        "MAE naive":         m["mae_naive"].mean(),
        "mejora vs naive %": (1 - m["mae_modelo"].mean() / m["mae_naive"].mean()) * 100,
        "MAE verano 2026":   m.loc["2026-07-01":, "mae_modelo"].mean(),
        "sesgo verano 2026": err[verano].mean(),
        "spearman":          m["spearman"].mean(),
        "regret compra K4":  m["regret_compra_k4"].mean(),
        "regret venta K4":   m["regret_venta_k4"].mean(),
    }

comparacion = pd.DataFrame(filas).T.round(2)
comparacion

In [ ]:
comparacion.index.name = "politica"
comparacion.to_csv(ROOT / "reports" / "comparacion_politicas_reentreno.csv")